## 1. Khám phá dữ liệu

In [1]:
# Thư viện
import pandas as pd
import re

In [2]:
df_users_raw = pd.read_csv('users_raw.csv')
df_posts_raw = pd.read_csv('posts_raw.csv')

In [3]:
print("Kích thước của df_users_raw:", df_users_raw.shape)
print("Kích thước của df_posts_raw:", df_posts_raw.shape)

Kích thước của df_users_raw: (80, 11)
Kích thước của df_posts_raw: (500, 11)


In [4]:
df_users_raw.head()

,user_id,user_name,country,language,account_type,followers_count,verified,join_date,age_group,gender,industry
0,U001,NGOC.TRINH446,DE,fre,media,47302,NaN,11/7/2023,18-24,f,SPORTS
1,U002,JOHN.SMITH 😀,france,VIET,PERSONAL,1087,0,19-05-2022,45 - 54,FEMALE,Technology
2,U003,news.today372 😀,Japan,NaN,Media,443789,FALSE,not_a_date,25 to 34,F,Food & Beverage
3,U004,movie_buff,Vietnam,ko,Influencer,157486,no,5/8/2024,45 - 54,M,education
4,U005,chef.tasty606,France,german,personal,2030,no,5/12/2019,25_34,NaN,Sports


In [5]:
df_posts_raw.head()

,post_id,platform,user_id,user_name,created_at,text,likes,comments,shares,sentiment_label,country
0,P00001,Fb,U035,hoa.le,22/02/2025 16:06,Feeling great today with #sale! Comme...,1395.0,436.0,26.0,pos,UK
1,P00002,youtube,U043,foodlover,2025/01/21 08:57:00,check out our new campaign #blackfriday on you...,2594.0,749.0,459.0,pos,Vietnam
2,P00003,Facebook,U019,sara.kim,03-01-2025,TOP 5 TIPS FOR BETTER RESULTS WITH BRAND X #BL...,NaN,393.0,NaN,Positive,DE
3,P00004,tiktok,U027,game.stream,2025-03-14 18:00:00,TOP 5 TIPS FOR BETTER RESULTS WITH ...,3992.0,533.0,429.0,POS,vietnam
4,P00005,X,U011,chef.tasty,26/03/2025 16:10,check out our new campaign #tiktoktrend on twi...,1870.0,198.0,293.0,Neutral,Vietnam


- Các cột **platform, country, sentiment_label** có nhiều cách viết khác nhau (hoa, thường, ký hiệu), gây thiếu đồng nhất.
- Các cột **created_at, join_date** không thống nhất định dạng thời gian và có giá trị lỗi như `not_a_date`.
- Các cột **followers_count, verified** lẫn kiểu dữ liệu (số, chuỗi, TRUE/FALSE, 0/1, NA), cần chuẩn hóa trước khi phân tích.

In [6]:
# Kiểu dl
print("\nThông tin kiểu dữ liệu của users")
print(df_users_raw.dtypes)
print("\nThông tin kiểu dữ liệu của posts")
print(df_posts_raw.dtypes)


Thông tin kiểu dữ liệu của users
user_id            object
user_name          object
country            object
language           object
account_type       object
followers_count    object
verified           object
join_date          object
age_group          object
gender             object
industry           object
dtype: object

Thông tin kiểu dữ liệu của posts
post_id             object
platform            object
user_id             object
user_name           object
created_at          object
text                object
likes              float64
comments           float64
shares             float64
sentiment_label     object
country             object
dtype: object


## 2. Làm sạch dữ liệu bài đăng (posts_raw)

In [7]:
df_posts = df_posts_raw.copy()

### Cột platform

In [8]:
# các gt duy nhất trong cột platform
print(df_posts['platform'].unique())

[' Fb ' 'youtube' 'Facebook' 'tiktok' 'X' 'twitter' ' twt ' ' insta '
 ' yt ' 'FACEBOOK' 'FB' 'YouTube' 'YOUTUBE' 'Instagram' 'TikTok' 'TIKTOK'
 'instagram' 'Twitter' ' Tik Tok ' 'IG' 'facebook' 'TWITTER']


In [9]:
# Bỏ khoảng trắng thừa, chuyển về chữ thường
df_posts['platform'] = df_posts['platform'].astype(str).str.strip().str.lower()
# Gom các biến thể về dạng chuẩn(lowercase)
platform_mapping_lower = {
    'fb': 'facebook',
    'facebook': 'facebook',
    'face book': 'facebook',

    'x': 'twitter',
    'twitter': 'twitter',
    'twt': 'twitter',

    'insta': 'instagram',
    'instagram': 'instagram',
    'ig': 'instagram',

    'tiktok': 'tiktok',
    'tik tok': 'tiktok',
    'tt': 'tiktok',

    'yt': 'youtube',
    'youtube': 'youtube',
    'you tube': 'youtube'
}
df_posts['platform'] = df_posts['platform'].replace(platform_mapping_lower)
# Chuyển về dạng chuẩn cuối cùng (Title Case)
title_map = {
    'facebook': 'Facebook',
    'twitter': 'Twitter',
    'instagram': 'Instagram',
    'tiktok': 'TikTok',
    'youtube': 'YouTube'
}

df_posts['platform'] = df_posts['platform'].map(title_map).fillna('Unknown')

print("Các giá trị duy nhất cột 'platform' sau chuẩn hóa:")
print(df_posts['platform'].unique())

Các giá trị duy nhất cột 'platform' sau chuẩn hóa:
['Facebook' 'YouTube' 'TikTok' 'Twitter' 'Instagram']


### Cột sentiment_label

In [10]:
print(df_posts['sentiment_label'].unique())
print(df_posts['sentiment_label'].value_counts(dropna=False))

['pos' 'Positive' 'POS' 'Neutral' 'Negative' 'positive' 'mixed' 'unknown'
 'neutral' 'NEU' 'neg' nan 'NEG' 'negative']
sentiment_label
Positive    56
pos         54
POS         51
neutral     47
NEU         47
positive    46
Neutral     45
Negative    35
NEG         28
negative    21
neg         20
NaN         20
mixed       15
unknown     15
Name: count, dtype: int64


In [ ]:
# Chuẩn hoá sentiment về lowercase + bỏ khoảng trắng
df_posts['sentiment_label'] = df_posts['sentiment_label'].astype(str).str.strip().str.lower()

# Gom nhóm các biến thể về chuẩn
sentiment_map = {
    'positive': 'positive',
    'pos': 'positive',
    'p': 'positive',

    'negative': 'negative',
    'neg': 'negative',
    'n': 'negative',

    'neutral': 'neutral',
    'neu': 'neutral'
}

df_posts['sentiment_label'] = df_posts['sentiment_label'].replace(sentiment_map)

# Gán các giá trị không xác định thành NaN (bao gồm cả chuỗi 'nan')
df_posts['sentiment_label'] = df_posts['sentiment_label'].replace(
    ['mixed', 'unknown', 'none', '', 'nan'],
    pd.NA
)

# Loại bỏ các dòng sentiment thiếu
df_posts = df_posts.dropna(subset=['sentiment_label'])

print("\nGiá trị sentiment sau khi chuẩn hóa:")
print(df_posts['sentiment_label'].unique())
print("\nSố lượng từng nhãn:")
print(df_posts['sentiment_label'].value_counts())


Giá trị sentiment sau khi chuẩn hóa:
['positive' 'neutral' 'negative']

Số lượng từng nhãn:
sentiment_label
positive    207
neutral     139
negative    104
Name: count, dtype: int64


Số lượng các giá trị không xác định ("mixed", "unknown", rỗng) = 50. các giá trị đó đã được gán thành NaN, sau đó em đã loại bỏ nó nhằm đảm bảo độ tin cậy cho các phân tích sau này liên quan đến phân bố sentiment của người dùng. Điều này giúp tránh việc phình to nhóm neutral và giảm sai lệch thống kê.

### cột country

In [12]:
print(df_posts['country'].unique())

['UK' 'Vietnam' 'DE' 'vietnam' 'Germany' 'U.K.' 'United Kingdom' 'TH'
 'Japan' 'Korea' 'VN' 'thailand' 'JP' 'Viet Nam' 'South Korea' 'KR'
 'Thailand' 'korea' 'germany' 'japan' 'uk' 'us' 'United States' 'USA'
 'U.S.A']


In [13]:
# Chuyển về lowercase + xóa khoảng trắng
df_posts['country'] = df_posts['country'].astype(str).str.strip().str.lower()

# Mapping đầy đủ dựa trên biến thể thực tế
country_map = {
    # Vietnam
    'vietnam': 'Vietnam',
    'viet nam': 'Vietnam',
    'vn': 'Vietnam',

    # USA
    'usa': 'USA',
    'u.s.a': 'USA',
    'us': 'USA',
    'united states': 'USA',

    # UK
    'uk': 'UK',
    'u.k.': 'UK',
    'united kingdom': 'UK',

    # Korea
    'korea': 'Korea',
    'south korea': 'Korea',
    'kr': 'Korea',

    # Japan
    'japan': 'Japan',
    'jp': 'Japan',

    # Thailand
    'thailand': 'Thailand',
    'th': 'Thailand',

    # Germany
    'germany': 'Germany',
    'de': 'Germany'
}

# Thay thế theo mapping
df_posts['country'] = df_posts['country'].replace(country_map)

# Xử lý các giá trị không hợp lệ
df_posts['country'] = df_posts['country'].replace(['nan', 'unknown', ''], pd.NA)

print("\nCác giá trị duy nhất của country sau chuẩn hóa:")
print(df_posts['country'].unique())
print(df_posts['country'].value_counts(dropna=False))


Các giá trị duy nhất của country sau chuẩn hóa:
['UK' 'Vietnam' 'Germany' 'Thailand' 'Japan' 'Korea' 'USA']
country
Vietnam     73
UK          68
Germany     68
Thailand    61
Japan       61
Korea       60
USA         59
Name: count, dtype: int64


### cột created_at

In [14]:
# Chuyển sang kiểu datetime (nhiều format)
df_posts['created_at'] = pd.to_datetime(
    df_posts['created_at'],
    errors='coerce',
    dayfirst=True
)

# Xóa các dòng không parse được thời gian
df_posts = df_posts.dropna(subset=['created_at'])

# Tạo thêm cột date (YYYY-MM-DD)
df_posts['date'] = df_posts['created_at'].dt.date.astype('datetime64[ns]')

# Tạo thêm cột hour (giờ trong ngày)
df_posts['hour'] = df_posts['created_at'].dt.hour.fillna(0).astype(int)

# Kiểm tra kết quả
print("\nSau khi chuẩn hóa created_at:")
print(df_posts[['created_at', 'date', 'hour']].head())
print(df_posts['created_at'].dtype)


Sau khi chuẩn hóa created_at:
            created_at       date  hour
0  2025-02-22 16:06:00 2025-02-22    16
4  2025-03-26 16:10:00 2025-03-26    16
5  2025-03-22 19:00:00 2025-03-22    19
9  2025-01-10 06:55:00 2025-01-10     6
11 2025-03-27 22:19:00 2025-03-27    22
datetime64[ns]


Tạo thêm cột hour = giờ trong ngày (0–23), không có giờ thì em thay bằng 0

In [15]:
# Chuẩn hóa các cột số likes, comments, shares

numeric_cols = ['likes', 'comments', 'shares']

for col in numeric_cols:
    # Chuyển kiểu về string -> loại khoảng trắng -> bỏ dấu phẩy
    df_posts[col] = df_posts[col].astype(str).str.strip().str.replace(',', '')

    # Thay các giá trị không hợp lệ thành NaN
    df_posts[col] = df_posts[col].replace(['', 'nan', 'na', 'null', 'None'], pd.NA)

    # Chuyển sang numeric, sai thì thành NaN
    df_posts[col] = pd.to_numeric(df_posts[col], errors='coerce')

    # NaN => gán = 0
    df_posts[col] = df_posts[col].fillna(0).astype(int)

# Tạo cột engagement
df_posts['engagement'] = (
    df_posts['likes'] + df_posts['comments'] + df_posts['shares']
)

print(df_posts[['likes', 'comments', 'shares', 'engagement']].head())
print(df_posts.dtypes)

    likes  comments  shares  engagement
0    1395       436      26        1857
4    1870       198     293        2361
5     244       327     147         718
9     234         0     265         499
11   2116       263     284        2663
post_id                    object
platform                   object
user_id                    object
user_name                  object
created_at         datetime64[ns]
text                       object
likes                       int32
comments                    int32
shares                      int32
sentiment_label            object
country                    object
date               datetime64[ns]
hour                        int32
engagement                  int32
dtype: object


### Xử lý text

In [16]:
# Tạo cột text_clean (lowercase, strip, bỏ nhiều khoảng trắng)
df_posts['text_clean'] = (
    df_posts['text']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

# Tạo cột hashtag_list: trích các hashtag (#...)
df_posts['hashtag_list'] = (
    df_posts['text']
    .astype(str)
    .apply(lambda x: re.findall(r'#\w+', x.lower()))
)

print(df_posts[['text', 'text_clean', 'hashtag_list']].head())

                                                 text  \
0       Feeling  great  today  with  #sale!  Comme...   
4   check out our new campaign #tiktoktrend on twi...   
5   Bạn đã thử sản phẩm mới chưa? #vlog @daily.vlo...   
9   TOP 5 TIPS FOR BETTER RESULTS WITH BRAND X #SU...   
11    FEELING GREAT TODAY WITH #BLACKFRIDAY! COMME...   

                                           text_clean    hashtag_list  
0   feeling great today with #sale! comment your t...         [#sale]  
4   check out our new campaign #tiktoktrend on twi...  [#tiktoktrend]  
5   bạn đã thử sản phẩm mới chưa? #vlog @daily.vlo...         [#vlog]  
9   top 5 tips for better results with brand x #su...       [#summer]  
11  feeling great today with #blackfriday! comment...  [#blackfriday]  


## Xử lý dữ liệu bảng users_raw

### Chuẩn hoá country, language, account_type

In [17]:
df_users = df_users_raw.copy()

In [18]:
print("Country:")
print(df_users_raw['country'].unique())

print("\nLanguage:")
print(df_users_raw['language'].unique())

print("\nAccount Type:")
print(df_users_raw['account_type'].unique())

print("\nTần suất account_type:")
print(df_users_raw['account_type'].value_counts(dropna=False))

Country:
['DE' 'france' 'Japan' 'Vietnam' 'France' 'United Kingdom' 'U.K.'
 'United States' 'JP' 'U.S.A' 'vietnam' 'VN' 'South Korea' 'Thailand'
 'Germany' 'TH' 'germany' 'korea' 'UK' 'Korea' 'japan' 'FR' 'GB' 'US'
 'thailand' 'KR' 'vn' 'us']

Language:
['fre' 'VIET' nan 'ko' 'german' 'thai' 'jp' 'ger' 'JA' 'french' 'TH' 'FR'
 'eng' 'fr' 'EN' 'de' 'DE' 'vietnamese' 'KO' 'japanese' 'korean' 'en' 'vi'
 'th']

Account Type:
['media' 'PERSONAL' 'Media' 'Influencer' 'personal' 'influencer'
 'Personal' 'publisher' 'user' 'individual' 'brand page' 'Brand' 'creator'
 nan 'Creator' 'brand' 'BRAND' 'MEDIA' 'CREATOR']

Tần suất account_type:
account_type
Personal      14
personal      13
user           9
PERSONAL       7
individual     5
NaN            4
media          4
publisher      3
influencer     3
Influencer     3
brand page     3
Media          3
Creator        2
BRAND          2
Brand          1
creator        1
brand          1
MEDIA          1
CREATOR        1
Name: count, dtype: int64

In [19]:
# COUNTRY
df_users['country'] = df_users['country'].astype(str).str.strip().str.lower()

country_map = {
    'vietnam': 'Vietnam', 'viet nam': 'Vietnam', 'vn': 'Vietnam',
    'usa': 'USA', 'u.s.a': 'USA', 'united states': 'USA', 'us': 'USA',
    'uk': 'UK', 'united kingdom': 'UK', 'u.k.': 'UK', 'gb': 'UK',
    'korea': 'Korea', 'south korea': 'Korea', 'kr': 'Korea',
    'japan': 'Japan', 'jp': 'Japan',
    'thailand': 'Thailand', 'th': 'Thailand',
    'germany': 'Germany', 'de': 'Germany', 'deutschland': 'Germany',
    'france': 'France', 'fr': 'France'
}

df_users['country'] = df_users['country'].replace(country_map)
df_users['country'] = df_users['country'].replace(['nan', 'unknown', ''], pd.NA)


# LANGUAGE

df_users['language'] = df_users['language'].astype(str).str.strip().str.lower()

language_map = {
    'english': 'en', 'eng': 'en', 'en': 'en', 'en-us': 'en',
    'viet': 'vi', 'vi': 'vi', 'vietnamese': 'vi',
    'korean': 'ko', 'ko': 'ko',
    'japanese': 'ja', 'ja': 'ja', 'jp': 'ja',
    'thai': 'th', 'th': 'th',
    'german': 'de', 'ger': 'de', 'de': 'de',
    'french': 'fr', 'fre': 'fr', 'fr': 'fr'
}

df_users['language'] = df_users['language'].replace(language_map)
df_users['language'] = df_users['language'].replace(['nan', 'unknown', ''], pd.NA)

# ACCOUNT TYPE
df_users['account_type'] = df_users['account_type'].astype(str).str.strip().str.lower()

account_type_map = {
    'user': 'personal', 'individual': 'personal', 'personal': 'personal',
    'brand': 'brand', 'brand page': 'brand',
    'influencer': 'influencer', 'infl': 'influencer',
    'media': 'media', 'publisher': 'media', 'news': 'media',
    'creator': 'creator', 'content creator': 'creator'
}

df_users['account_type'] = df_users['account_type'].replace(account_type_map)
df_users['account_type'] = df_users['account_type'].replace(['nan', 'unknown', ''], pd.NA)

print("Country:", df_users['country'].unique())
print("Language:", df_users['language'].unique())
print("Account type:", df_users['account_type'].unique())

Country: ['Germany' 'France' 'Japan' 'Vietnam' 'UK' 'USA' 'Korea' 'Thailand']
Language: ['fr' 'vi' <NA> 'ko' 'de' 'th' 'ja' 'en']
Account type: ['media' 'personal' 'influencer' 'brand' 'creator' <NA>]


In [20]:
print("Số giá trị thiếu theo từng cột:")
print(df_users[['country', 'language', 'account_type']].isna().sum())

Số giá trị thiếu theo từng cột:
country          0
language        14
account_type     4
dtype: int64


In [ ]:
# Xử lý NaN của language = điền language dựa trên country
# Nếu thiếu nhiều thì fill theo country nha
country_to_lang = {
    'Vietnam': 'vi',
    'USA': 'en',
    'UK': 'en',
    'Korea': 'ko',
    'Japan': 'ja',
    'Thailand': 'th',
    'Germany': 'de',
    'France': 'fr'
}

df_users['language'] = df_users.apply(
    lambda row: country_to_lang[row['country']] if pd.isna(row['language']) and row['country'] in country_to_lang else row['language'],
    axis=1
)
# Loại bỏ các dòng thiếu account_type
df_users = df_users.dropna(subset=['account_type'])

In [22]:
print(df_users['language'].unique())
print(df_users['account_type'].unique())

['fr' 'vi' 'ja' 'ko' 'de' 'th' 'en']
['media' 'personal' 'influencer' 'brand' 'creator']


### Chuẩn hoá followers_count

In [23]:
# Chuẩn hóa followers_count

df_users['followers_count'] = (
    df_users['followers_count']
    .astype(str)
    .str.strip()
    .str.replace(',', '')
)

# Các giá trị lỗi -> NaN
df_users['followers_count'] = df_users['followers_count'].replace(
    ['', 'na', 'null', 'none', 'nan'], pd.NA
)

# Convert sang numeric, lỗi -> NaN
df_users['followers_count'] = pd.to_numeric(
    df_users['followers_count'], errors='coerce'
)

# NaN => gán 0
df_users['followers_count'] = df_users['followers_count'].fillna(0).astype(int)


print(df_users[['followers_count']].head())
print(df_users['followers_count'].dtype)

   followers_count
0            47302
1             1087
2           443789
3           157486
4             2030
int32


### Chuẩn hóa verified

In [24]:
print("Giá trị unique trong verified:")
print(df_users_raw['verified'].unique())

Giá trị unique trong verified:
[nan '0' 'FALSE' 'no' '1' 'TRUE']


In [25]:
# Chuẩn hóa cột verified

df_users['verified'] = df_users['verified'].astype(str).str.strip().str.lower()

verified_map = {
    'true': True,
    '1': True,
    'false': False,
    '0': False,
    'no': False
}

df_users['verified'] = df_users['verified'].replace(verified_map)

# NaN hoặc giá trị không map được → False
df_users['verified'] = df_users['verified'].fillna(False)

# Đảm bảo kiểu bool
df_users['verified'] = df_users['verified'].astype(bool)

print(df_users['verified'].unique())
print(df_users['verified'].value_counts())
print(df_users['verified'].dtype)

[ True False]
verified
False    47
True     29
Name: count, dtype: int64
bool


Dữ liệu cột verified ban đầu tồn tại nhiều biến thể khác nhau như **‘TRUE’, ‘FALSE’, ‘1’, ‘0’, ‘no’, và giá trị thiếu**. Tất cả được chuẩn hoá về kiểu Boolean True/False. Các giá trị thiếu được gán False vì trạng thái chưa xác minh là phổ biến nhất và phù hợp với ngữ nghĩa của dữ liệu mạng xã hội.

### Chuẩn hoá join_date:

In [26]:
df_users['join_date'] = pd.to_datetime(
    df_users['join_date'],
    errors='coerce',
    dayfirst=True
)

print(df_users['join_date'].head())
print(df_users['join_date'].dtype)
print("Số dòng join_date bị NaT:", df_users['join_date'].isna().sum())

0   2023-07-11
1          NaT
2          NaT
3   2024-08-05
4   2019-12-05
Name: join_date, dtype: datetime64[ns]
datetime64[ns]
Số dòng join_date bị NaT: 45


Các giá trị "not_a_date" hoặc không parse được, em gán là NaT. Vì tỷ lệ thiếu rất cao nên em giữ NaT để đảm bảo tính trung thực dữ liệu, và chỉ loại NaT khi phân tích theo thời gian.

### Chuẩn hoá age_group & gender:

In [27]:
print("Unique age_group:")
print(df_users_raw['age_group'].unique())

print("\nUnique gender:")
print(df_users_raw['gender'].unique())

Unique age_group:
['18-24' '45 - 54' '25 to 34' '25_34' '25 - 34' '18 - 24' '35_44' '45_54'
 '35 - 44' '45-54' '45 to 54' nan '25-34' '35-44' '18_24' '18 to 24'
 '35 to 44']

Unique gender:
['f' 'FEMALE' 'F' 'M' nan 'non-binary' 'other' 'female' 'OTHER' 'Female'
 'm' 'male' 'MALE' 'Male' 'Other']


In [28]:
# AGE_GROUP
df_users['age_group'] = (
    df_users['age_group'].astype(str)
    .str.lower()
    .str.replace(' ', '')
    .str.replace('_', '-')
    .str.replace('to', '-')
)

age_map = {
    '18-24': '18-24',
    '25-34': '25-34',
    '35-44': '35-44',
    '45-54': '45-54'
}

df_users['age_group'] = df_users['age_group'].replace(age_map)
df_users['age_group'] = df_users['age_group'].replace(['nan', ''], pd.NA)


# GENDER
df_users['gender'] = df_users['gender'].astype(str).str.strip().str.lower()

gender_map = {
    'm': 'male', 'male': 'male',
    'f': 'female', 'female': 'female',
    'non-binary': 'other', 'other': 'other'
}

df_users['gender'] = df_users['gender'].replace(gender_map)

# Loại giá trị 'nan' dạng string -> NaN
df_users['gender'] = df_users['gender'].replace(['nan', 'unknown', 'none', ''], pd.NA)

# NaN -> other
df_users['gender'] = df_users['gender'].fillna('other')

print("Age group unique:", df_users['age_group'].unique())
print("Gender unique:", df_users['gender'].unique())

Age group unique: ['18-24' '45-54' '25-34' '35-44' <NA>]
Gender unique: ['female' 'male' 'other']


### Cột industry

In [29]:
df_users['industry'] = df_users['industry'].astype(str)

# Loại khoảng trắng thừa và đồng nhất chữ
df_users['industry'] = (
    df_users['industry']
    .str.strip()
    .str.title()
)

# Giá trị rỗng hoặc 'Nan' thành NaN thực sự
df_users['industry'] = df_users['industry'].replace(['', 'Nan', 'None', 'Null'], pd.NA)

print("Industry unique:", df_users['industry'].unique())
print("Số NaN trong industry:", df_users['industry'].isna().sum())

Industry unique: ['Sports' 'Technology' 'Food & Beverage' 'Education' 'Entertainment'
 'Travel' 'Beauty & Fashion' 'Retail  🔥' <NA> 'Food & Beverage  🔥'
 'Technology  😀' 'Retail 😀' 'Retail' 'Food & Beverage ❤️' 'Technology 😀'
 'Finance' 'Sports 😀' 'Sports ❤️' 'Beauty & Fashion  ⭐' 'Entertainment  ⭐'
 'Education  😀' 'Finance 🔥' 'Retail  ⭐' 'Education  ⭐']
Số NaN trong industry: 3


In [30]:
print("\nMissing values:")
print(df_users.isna().sum())


Missing values:
user_id             0
user_name           2
country             0
language            0
account_type        0
followers_count     0
verified            0
join_date          45
age_group           5
gender              0
industry            3
dtype: int64


In [31]:
df_users['user_name'] = df_users['user_name'].fillna('Unknown')
print(df_users['user_name'].isna().sum())

0


## Kết hợp dữ liệu & xuất file sạch

In [32]:
# Kết hợp bảng posts và users bằng INNER JOIN trên user_id
df_merged = pd.merge(
    df_posts,
    df_users,
    on='user_id',
    how='inner'
)

# Kiểm tra số lượng dòng trước & sau join
print("Số dòng df_posts:", len(df_posts))
print("Số dòng df_users:", len(df_users))
print("Số dòng df_merged:", len(df_merged))

# Số post không khớp user
missing_posts = len(df_posts) - len(df_merged)
print("\nSố post không khớp user_id:", missing_posts)

Số dòng df_posts: 116
Số dòng df_users: 76
Số dòng df_merged: 110

Số post không khớp user_id: 6


In [33]:
# Xuất dữ liệu sạch

df_posts.to_csv('posts_clean_22174600025.csv', index=False)
df_users.to_csv('users_clean_22174600025.csv', index=False)
df_merged.to_csv('social_media_merged_22174600025.csv', index=False)
print("Xuất file thành công")

Xuất file thành công


--- Ghi chú tổng kết xử lý dữ liệu ---

- Dữ liệu ban đầu tồn tại nhiều giá trị không đồng nhất ở các cột phân loại như 
platform, country, language, account_type, sentiment_label (khác biệt hoa/thường, 
viết tắt, ký hiệu), gây khó khăn cho việc nhóm, lọc và tổng hợp dữ liệu.

- Các cột thời gian và số (created_at, join_date, likes, followers_count, …) 
không thống nhất định dạng, đồng thời chứa các giá trị lỗi hoặc không hợp lệ 
như not_a_date, NA, null.

--- Cách xử lý và lý do lựa chọn ---

- Các biến phân loại được chuẩn hóa thông qua mapping về tập giá trị chuẩn nhằm 
đảm bảo tính nhất quán khi phân tích và trực quan hóa trong Power BI.

- Những dòng dữ liệu không thể sử dụng cho phân tích (thời gian không parse được, 
sentiment không xác định, thiếu account_type) được loại bỏ để đảm bảo chất lượng dữ liệu đầu vào.

- Đối với các cột số và boolean, các giá trị không hợp lệ được chuyển về 0 hoặc False 
nhằm tránh lỗi tính toán và đảm bảo dashboard hoạt động ổn định.
